# v6 split-KV decode — the gate (vast.ai T4)

Standalone, **Run All** top-to-bottom. No v1–v5 bench/profiling — just the v6 decode gate.

Decode = `N_q = 1`: v1–v5 launch `(1, B·H)` blocks and starve the SMs. v6 splits the KV axis across
blocks (Flash-Decoding) — each block emits an unnormalized `(O, m, ℓ)` partial, a merge kernel does
the log-sum-exp combine. FP16-in / FP32-accum (no tensor cores: the matmuls are M=1).

Gate (counter-free — `ncu` is blocked on vast.ai): **(1) correctness vs SDPA** + **(2) decode bench**
(µs/token, % HBM bandwidth, speedup vs a naive `seqlen_q=1` loop). **Pick a T4 GPU + CUDA-devel image.**

## 0. Dependencies + GPU (venv-safe)
Installs into **this kernel's** Python (works with or without a venv) and prepends the kernel's
`bin/` to PATH so every later `!python -m …` cell resolves. Torch is only installed if missing.

In [1]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

try:
    import torch  # already on the image? keep it, don't churn the version
except ImportError:
    pip('torch', extra=('--index-url', 'https://download.pytorch.org/whl/cu124'))
pip('ninja', 'pytest', 'numpy')

# vast.ai venv fix: !-cells spawn a bare shell without the venv on PATH -> `python` not found.
os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')

import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap --format=csv
!which python && python -c "import torch; print('shell python sees torch', torch.__version__)"

torch 2.6.0+cu124 | cuda 12.4 | cap (7, 5)
name, compute_cap
Tesla T4, 7.5
/venv/main/bin/python
shell python sees torch 2.6.0+cu124


## 1. Get the repo
Clones if absent, otherwise pulls the latest `main`. Adds the repo root to `sys.path`. Safe to re-run.

In [2]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

Updating 9550953..3cab80a
Fast-forward
 notebooks/colab_bootstrap.ipynb |  2 +-
 notebooks/v6_decode_gate.ipynb  | 93 +++++++++++++++++++++++++++++++++++++++++
 2 files changed, 94 insertions(+), 1 deletion(-)
 create mode 100644 notebooks/v6_decode_gate.ipynb
cwd /flashattention-cuda


From https://github.com/gkienpham-cmd/flashattention-cuda
 * branch            main       -> FETCH_HEAD
   9550953..3cab80a  main       -> origin/main


## 2. Roofline first (decode `AI = 2/b`, before running)
At `N_q = 1`: work = 4·N_k·d FLOPs, traffic = 2·N_k·d·b bytes (read K and V) ⇒ `AI = 2/b`,
**independent of N_k** — pure HBM-bound, far below the fp16 ridge (~203). FP16 KV (`b = 2`) ⇒ `AI = 1.0`.
Expect every row `HBM`. `predict.py` only takes square shapes, so we call the model directly.

In [3]:
from roofline.archs import get_arch
from roofline.model import estimate
arch = get_arch('sm_75')
print(f"{'shape q/kv':>14} | {'limiter':>7} | {'AI':>4} | {'ridge':>6} | {'t_hbm floor':>11}")
for d in (64, 128):
    for Nk in (2048, 8192, 16384):
        e = estimate(arch, B=1, H=8, N_q=1, N_k=Nk, d=d, precision='fp16', materialize_s=False)
        print(f"{f'1x8x1x{d}/{Nk}':>14} | {e.limiter.upper():>7} | {e.arithmetic_intensity:4.2f} | "
              f"{e.ridge:6.1f} | {e.t_hbm*1e3:8.4f}ms")

    shape q/kv | limiter |   AI |  ridge | t_hbm floor
 1x8x1x64/2048 |     HBM | 1.00 |  203.1 |   0.0131ms
 1x8x1x64/8192 |     HBM | 1.00 |  203.1 |   0.0524ms
1x8x1x64/16384 |     HBM | 1.00 |  203.1 |   0.1049ms
1x8x1x128/2048 |     HBM | 1.00 |  203.1 |   0.0262ms
1x8x1x128/8192 |     HBM | 1.00 |  203.1 |   0.1049ms
1x8x1x128/16384 |     HBM | 1.00 |  203.1 |   0.2097ms


## 3. Build v6 (JIT)
First call compiles with nvcc (~1 min), cached after. The clean-up guard removes any interrupted
build dir (a version-stamped dir with no `.so`) that would otherwise make torch skip the rebuild.

In [4]:
import glob, os, shutil
for d in glob.glob(os.path.expanduser('~/.cache/torch_extensions/*/fa_v6_splitkv')):
    if not glob.glob(os.path.join(d, '*.so')):
        shutil.rmtree(d, ignore_errors=True); print('cleaned stale build:', d)
from bindings.load import build_kernel
mod = build_kernel('v6_splitkv')
print('built:', mod)

Using /root/.cache/torch_extensions/py312_cu124 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py312_cu124/fa_v6_splitkv/build.ninja...
/venv/main/lib/python3.12/site-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module fa_v6_splitkv...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


ninja: no work to do.
built: <module 'fa_v6_splitkv' from '/root/.cache/torch_extensions/py312_cu124/fa_v6_splitkv/fa_v6_splitkv.so'>


Loading extension module fa_v6_splitkv...


## 4. Correctness gate — v6 vs SDPA  *(Gate 1 of 2)*
`-k v6_splitkv` runs **both** the square SHAPES (where `choose_splits → 1`, so v6 reduces to plain
attention) **and** `test_v6_splitkv_decode` (`N_q=1`, `N_k ∈ {4096, 8192, 8190}`, d 64/128, causal both
ways) — the only test that drives the real cross-split LSE merge. Tolerance 2e-2 (FP16-in, like v5).
**Must be all-green.**

In [5]:
!python -m pytest tests/test_correctness.py -k v6_splitkv -q

.........................                                                [100%]
=============================== warnings summary ===============================
tests/test_correctness.py::test_matches_sdpa[False-1-4-128-64-v6_splitkv]
  /venv/main/lib/python3.12/site-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
  If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
    warnings.warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
25 passed, 75 deselected, 1 warning in 2.17s


## 5. Decode benchmark (the deliverable)
`--decode`: query length collapses to 1, the swept N is the KV length (2048/8192/16384 × d 64/128).
Columns: **µs/tok**, **%HBM** (fraction of the 320 GB/s peak the K+V read achieves), **vs sdpa**, and
**vs naive** (v5 run at `N_q=1` — the `1×BH` single-block-per-head loop v6 must beat). Bigger `vs naive`
and higher `%HBM` at large `N_k` = the win. Paste the rows into `docs/results.md` Step 6.

In [6]:
!python -m bench.harness --backend v6_splitkv --decode
print()
!python -m bench.harness --backend v6_splitkv --decode --causal

# device: Tesla T4 (sm_75)  clock~-1/-1MHz  backend=v6_splitkv  precision=fp32  causal=False  decode=True
#    shape(q x kv) |    ours p50/max ms |   us/tok |   %HBM |  vs sdpa | vs naive | roofline
Using /root/.cache/torch_extensions/py312_cu124 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py312_cu124/fa_v6_splitkv/build.ninja...
/venv/main/lib/python3.12/site-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module fa_v6_splitkv...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)
ninja: no work to do.
Loading extension module fa_v6_splitkv...
Using /root/.cache/torch_extensions/py312_cu124 as PyTorch extensions root...
Creating extension di